In [0]:
%python
import subprocess
import os
import requests

# Generate key
for f in ["/tmp/brev_key", "/tmp/brev_key.pub"]:
    if os.path.exists(f):
        os.remove(f)

subprocess.run('ssh-keygen -t ed25519 -f /tmp/brev_key -N "" -q', shell=True, check=True)
os.chmod("/tmp/brev_key", 0o600)

with open("/tmp/brev_key") as f:
    private_key = f.read()
with open("/tmp/brev_key.pub") as f:
    public_key = f.read()

# Store in Databricks secrets via API
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
headers = {"Authorization": f"Bearer {token}"}

for key_name, key_value in [("ssh_private_key", private_key), ("ssh_public_key", public_key)]:
    resp = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers=headers,
        json={
            "scope": "brev",
            "key": key_name,
            "string_value": key_value
        }
    )
    print(f"{key_name}: {resp.status_code}")

# Clean up
os.remove("/tmp/brev_key")
os.remove("/tmp/brev_key.pub")

print("Keys stored in secrets.")

In [0]:
# Brev Enironment Setup

# use -> databricks secrets put-secret brev brev_credentials --string-value "$(cat ~/.brev/credentials.json)"
# (https://docs.databricks.com/aws/en/security/secrets/?language=Databricks%C2%A0CLI)

# Get secrets from Databricks
# Three scopes: |brev:              | wandb:     | databricks: 
#               |  -token           |   -token   |   - pat (personal access token)
#               |  -instance        |            |
#               |  -ssh_public_key  |            |
#               |  -ssh_private_key |            |
#               |  -dataset_name    |            |
#               |  -max_steps       |            |
#               |  -save_steps      |            |

# print the value of a secret from a scope: databricks secrets get-secret <scope-name> <key-name> | jq -r .value | base64 --decode

brev_credentials = dbutils.secrets.get(scope="brev", key="brev_credentials")
brev_instance = dbutils.secrets.get(scope="brev", key="instance")
dataset_name = dbutils.secrets.get(scope="brev", key="dataset_name")
max_steps = dbutils.secrets.get(scope="brev", key="max_steps")
save_steps = dbutils.secrets.get(scope="brev", key="save_steps")

ssh_pub_key = dbutils.secrets.get(scope="brev", key="ssh_public_key")
ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

wandb = dbutils.secrets.get(scope="wandb", key="token")

pat = dbutils.secrets.get(scope="databricks", key="pat") # pe cand da cristi token u

host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None) # ca sa luam datasetu cu wget

import os
os.environ['BREV_CREDENTIALS_JSON'] = brev_credentials
os.environ['BREV_INSTANCE_NAME'] = brev_instance

os.environ['SSH_PUB_KEY'] = ssh_pub_key
os.environ['SSH_PRIV_KEY'] = ssh_priv_key

os.environ['WANDB_API_KEY'] = wandb
os.environ['SAVE_STEPS'] = save_steps
os.environ['MAX_STEPS'] = max_steps

# Dataset
os.environ['DATASET_PATH'] = '/Volumes/workspace/default/datasets' 
os.environ['DATASET_NAME'] = dataset_name

# Modality files
os.environ['MODALITY_FILES_PATH'] = '/Volumes/workspace/default/modality_files'
os.environ['MODALITY_JSON'] = 'modality.json'
os.environ['MODALITY_PY'] = 'so100_top_wrist_config.py'
 
os.environ['DATABRICKS_TOKEN'] = pat
os.environ['DATABRICKS_HOST'] = host

In [0]:
%sh
set -euo pipefail # bash safety net

INSTALL_DIR="$HOME/.local/bin"
mkdir -p "$INSTALL_DIR"

# Download latest Brev CLI release matching this machine
OS="$(uname -s | tr '[:upper:]' '[:lower:]')" # ex: "linux"
ARCH="$(uname -m)"                            # ex: x86_64"

# normalization case to match the naming convention used in Brev's GitHub release assets
case "$ARCH" in                               
  x86_64) ARCH="amd64" ;;
  aarch64|arm64) ARCH="arm64" ;;
esac

# Download the latest Brev CLI install script
curl -fsSL https://raw.githubusercontent.com/brevdev/brev-cli/main/bin/install-latest.sh -o /tmp/install-brev.sh

# Add permissions
chmod +x /tmp/install-brev.sh

# Run the install script
/tmp/install-brev.sh

export PATH="$HOME/.local/bin:$PATH"

which brev
brev --version || true

echo "=== Restoring Brev credentials from secret (no more 15-min token) ==="
mkdir -p "$HOME/.brev"; chmod 700 "$HOME/.brev"
python3 -c "import os; open(os.path.expanduser('~/.brev/credentials.json'),'w').write(os.environ['BREV_CREDENTIALS_JSON'])"
chmod 600 "$HOME/.brev/credentials.json"

echo "=== Verifying brev auth (auto-refreshes the access token from the stored refresh token) ==="
brev ls

In [0]:
%sh
set -euo pipefail

export PATH="$HOME/.local/bin:$PATH"

# Refresh so ~/.brev/ssh_config has the latest IP/user
brev refresh

CFG="$HOME/.brev/ssh_config"

# Extract Hostname and User from the block matching $BREV_INSTANCE_NAME
awk -v host="$BREV_INSTANCE_NAME" '
  $1 == "Host" && $2 == host {found=1; next}
  $1 == "Host" && found {exit}
  found && $1 == "Hostname" {print $2 > "/tmp/brev_ip.txt"}
  found && $1 == "User" {print $2 > "/tmp/brev_ssh_user.txt"}
' "$CFG"

BREV_IP="$(cat /tmp/brev_ip.txt | tr -d '[:space:]')"
SSH_USER="$(cat /tmp/brev_ssh_user.txt | tr -d '[:space:]')"

echo "Brev IP: $BREV_IP"
echo "SSH user: $SSH_USER"

In [0]:
%python
import os
import requests
from pathlib import Path

brev_ip_path = Path("/tmp/brev_ip.txt")
ssh_user_path = Path("/tmp/brev_ssh_user.txt")

# Read values from temp files
brev_ip = brev_ip_path.read_text().strip().splitlines()[-1]
ssh_user = ssh_user_path.read_text().strip().splitlines()[-1]

# Put into env for this notebook context
os.environ["BREV_INSTANCE_IP"] = brev_ip
os.environ["SSH_USER"] = ssh_user

print(f"IP: {brev_ip}")
print(f"SSH_USER: {ssh_user}")

# Save both to Databricks secrets
secrets_to_save = {
    "brev_ip": brev_ip,
    "ssh_user": ssh_user,
}

for key, value in secrets_to_save.items():
    resp = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers=headers,
        json={
            "scope": "brev",
            "key": key,
            "string_value": value,
        },
    )
    print(f"{key}: {resp.status_code}")

# Remove temp files
for path in [brev_ip_path, ssh_user_path]:
    try:
        path.unlink()
        print(f"Removed {path}")
    except FileNotFoundError:
        pass

In [0]:
%python
from pathlib import Path

ssh_pub_key = dbutils.secrets.get(scope="brev", key="ssh_public_key").strip()

Path("/tmp/brev_pipeline_key.pub").write_text(ssh_pub_key + "\n")

print("Loaded public key from Databricks secrets.")
print("Public key file: /tmp/brev_pipeline_key.pub")

In [0]:
%sh
set -euo pipefail

SSH_PUB_KEY_B64="$(cat /tmp/brev_pipeline_key.pub | base64 | tr -d '\n')"

ssh -F "$HOME/.brev/ssh_config" \
  -o ControlMaster=no \
  -o ControlPath=none \
  -o RequestTTY=no \
  -o StrictHostKeyChecking=no \
  "$BREV_INSTANCE_NAME" "bash -s" << EOF
set -euo pipefail

mkdir -p "\$HOME/.ssh"
chmod 700 "\$HOME/.ssh"

touch "\$HOME/.ssh/authorized_keys"
chmod 600 "\$HOME/.ssh/authorized_keys"

PUB_KEY="\$(printf '%s' "$SSH_PUB_KEY_B64" | base64 -d)"

if ! grep -qxF "\$PUB_KEY" "\$HOME/.ssh/authorized_keys"; then
  printf '%s\n' "\$PUB_KEY" >> "\$HOME/.ssh/authorized_keys"
fi

echo "KEY_INSTALLED"
EOF